# Module 8 — Temporal Modes

Three modes applied universally: **historical / early / current**.

Early mode blends historical (3-yr smoothed) with current data. Current weight ramps from 0 at 15 games to 50% at 50 games.

**Delta** = current − historical. Positive = breakout signal. Negative = decline signal.

In [ ]:
import sys
sys.path.insert(0, "../modules")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from temporal import (
    determine_mode, blend_weights, compute_historical_smooth,
    apply_temporal_mode, compute_delta, process_metrics, apply_temporal_to_df,
)
%matplotlib inline

In [ ]:
# Mode determination examples
for pa, g in [(5,3),(60,25),(50,30),(250,70),(400,130)]:
    mode = determine_mode(pa, g)
    hw, cw = blend_weights(g)
    print(f"PA={pa:4d}  G={g:3d}  mode={mode:12s}  hist_w={hw:.3f}  curr_w={cw:.3f}")

In [ ]:
# Visualise early-mode blend weight ramp
games_range = range(15, 51)
hist_weights = [blend_weights(g)[0] for g in games_range]
curr_weights = [blend_weights(g)[1] for g in games_range]
fig, ax = plt.subplots(figsize=(9,4))
ax.fill_between(games_range, hist_weights, alpha=0.6, label="Historical weight", color="steelblue")
ax.fill_between(games_range, curr_weights, alpha=0.6, label="Current weight",    color="coral")
ax.set_xlabel("Games Played")
ax.set_ylabel("Weight")
ax.set_title("Early-Mode Blend Weight Ramp")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Historical smoothing example
season_vals = {2023: 70.0, 2022: 62.0, 2021: 58.0}
smooth = compute_historical_smooth(season_vals, reference_season=2023)
print(f"Season values: {season_vals}")
print(f"Smoothed (reference=2023): {smooth:.2f}")
print(f"Expected: {70.0*0.50 + 62.0*0.30 + 58.0*0.20:.2f}")

In [ ]:
# Full temporal profile for a player across metrics
current_metrics  = {"xwOBA_pct": 78.0, "K_pct_pct": 40.0, "BB_pct_pct": 72.0}
historical_metrics = {"xwOBA_pct": 65.0, "K_pct_pct": 48.0, "BB_pct_pct": 60.0}

for pa, games, label in [(8, 8, "Spring Training"), (90, 30, "Early Season"), (350, 90, "Full Season")]:
    r = process_metrics(pa, games, current_metrics, historical_metrics)
    print(f"
{label} (PA={pa}, G={games}, mode={r["xwOBA_pct"]["mode"]}):")
    for metric, vals in r.items():
        print(f"  {metric:<14}: adj={vals["adjusted"]:.1f}  delta={vals["delta"]:+.1f}  hist_w={vals["hist_weight"]:.2f}  curr_w={vals["curr_weight"]:.2f}")